
# Unified scaffold-aware train/test split pipeline

이 노트북은 기존처럼 여러 단계를 따로 거치지 않고, 아래를 **한 번에** 처리합니다.

1. 원본 canonical analysis 로드  
2. count-binned functional group rule 생성  
3. scaffold group 생성  
4. scaffold-aware 80:20 train/test split  
5. 엔드포인트별 **추천 union feature set**이 포함된 train/test 파일 저장  

최종적으로 각 엔드포인트마다 아래 3개만 바로 쓰면 됩니다.
- `*_recommended_union_train.csv`
- `*_recommended_union_test.csv`
- `*_recommended_union_with_split.csv`

`feature_manifest.json`에는 compact / extended feature 목록도 함께 저장됩니다.


In [1]:

# 사용자 설정
RAW_ZIP = "/mnt/data/fg_descriptor_analysis.zip"
RULE_HIGHLIGHTS = "/mnt/data/fg_count_threshold_highlights.csv"
OUTDIR = "/mnt/data/unified_train_test_pipeline_outputs"
RANDOM_STATE = 42
TEST_SIZE = 0.20

# QM descriptor 옵션 (없으면 그대로 False 유지)
USE_QM = False
QM_FILE = None
MERGE_KEY = "No"
QM_COLS = None


In [2]:

from __future__ import annotations

import io
import json
import zipfile
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdMolHash
from sklearn.model_selection import StratifiedGroupKFold
from IPython.display import display


In [3]:

ID_COLS = ["No", "SMILES", "label"]
META_COLS = [
    "murcko_scaffold",
    "scaffold_group",
    "scaffold_group_type",
    "scaffold_split_80_20",
]

ENDPOINTS: Dict[str, Dict[str, object]] = {
    "ames": {
        "raw_name": "ames_combine",
        "compact": [
            "scaffold_group_type",
            "bb_n_genotox_alerts",
            "aromatic_ring_count",
            "fraction_csp3",
            "rot_bonds",
            "tpsa",
            "rule_bb_sa5_nitro_aromatic_ge2",
            "rule_bb_sa10_n_nitroso_ge1",
            "rule_epoxide_ge1",
            "rule_hydrazine_like_ge1",
            "rule_ester_ge2",
            "rule_sulfone_ge2",
            "rule_bb_sa47_n_alkylcarboxylic_acid_ge1",
        ],
        "extended_extra": [
            "fg_BB_SA5_nitro_aromatic_present",
            "fg_aniline_present",
            "fg_heteroaromatic_ring_present",
            "fg_fused_ring_like_present",
            "fg_epoxide_present",
            "fg_BB_SA10_N_nitroso_present",
            "fg_hydrazine_like_present",
            "fg_BB_SA2c_aromatic_amine_general_present",
            "fg_ester_present",
            "fg_ether_present",
            "fg_carboxylic_acid_present",
            "fg_sulfone_present",
            "fg_sulfonic_acid_or_sulfonate_present",
            "rule_epoxide_ge2",
            "rule_fluoro_ge5",
            "bb_n_alerts",
            "fg_BB_SA2_primary_aromatic_amine_present",
            "fg_BB_SA40_alkyl_halides_nongx_present",
            "has_formal_charge",
            "ring_count",
            "heavy_atom_count",
        ],
    },
    "invitro": {
        "raw_name": "invitro_pre",
        "compact": [
            "scaffold_group_type",
            "bb_n_genotox_alerts",
            "rot_bonds",
            "logp",
            "fraction_csp3",
            "mw",
            "rule_bb_sa2_primary_aromatic_amine_ge1",
            "rule_aniline_ge1",
            "rule_bb_sa20_ab_unsat_carbonyl_ge1",
            "rule_bb_sa47_n_alkylcarboxylic_acid_ge1",
            "rule_ester_eq1",
        ],
        "extended_extra": [
            "fg_aniline_present",
            "fg_halogen_any_present",
            "fg_BB_SA2c_aromatic_amine_general_present",
            "fg_BB_SA20_ab_unsat_carbonyl_present",
            "fg_BB_SA47_n_alkylcarboxylic_acid_present",
            "rule_bb_sa2_primary_aromatic_amine_ge2",
            "rule_epoxide_ge1",
            "rule_alcohol_ge2",
            "bb_n_alerts",
            "fg_BB_SA2_primary_aromatic_amine_present",
            "fg_BB_SA40_alkyl_halides_nongx_present",
            "fg_BB_SA18_sulphonate_ester_present",
        ],
    },
    "invivo": {
        "raw_name": "invivo_pre",
        "compact": [
            "scaffold_group_type",
            "mw",
            "hbd",
            "logp",
            "heteroatom_count",
            "rot_bonds",
            "rule_epoxide_ge1",
            "rule_organometal_like_ge1",
            "rule_alcohol_ge1",
            "rule_fused_ring_like_ge1",
        ],
        "extended_extra": [
            "fg_epoxide_present",
            "fg_organometal_like_present",
            "fg_fused_ring_like_present",
            "fg_alcohol_present",
            "rule_epoxide_ge2",
            "rule_organometal_like_ge2",
            "contains_heavy_metal",
            "contains_metal",
            "fg_BB_SA16b_aziridine_present",
        ],
    },
}


In [4]:

def dedupe_keep_order(cols: Sequence[str]) -> List[str]:
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def rule_col_name(rule_row: pd.Series) -> str:
    base = str(rule_row["feature_clean"])
    mode = "ge" if rule_row["mode"] == "ge" else "eq"
    thr_val = float(rule_row["threshold"])
    thr = str(int(thr_val)) if thr_val.is_integer() else str(thr_val).replace(".", "p")
    safe = (
        base.lower()
        .replace("BB_", "bb_")
        .replace(" ", "_")
        .replace("-", "_")
        .replace("/", "_")
    )
    safe = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in safe)
    while "__" in safe:
        safe = safe.replace("__", "_")
    return f"rule_{safe}_{mode}{thr}"


def apply_rule(series: pd.Series, mode: str, threshold: float) -> pd.Series:
    s = series.fillna(0)
    if mode == "ge":
        return (s >= threshold).astype(int)
    if mode == "eq":
        return (s == threshold).astype(int)
    raise ValueError(f"Unsupported mode: {mode}")


def make_scaffold_group(smiles: str, murcko: object) -> Tuple[str, str]:
    murcko_str = None if pd.isna(murcko) else str(murcko).strip()
    if murcko_str:
        return murcko_str, "murcko"
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return f"ACY_INVALID::{smiles}", "acyclic_singleton"
    anon = rdMolHash.MolHash(mol, rdMolHash.HashFunction.AnonymousGraph)
    if anon:
        return f"ACY_{anon}", "acyclic_anonymous_graph"
    return f"ACY_SINGLE::{smiles}", "acyclic_singleton"


def choose_best_holdout(
    df: pd.DataFrame,
    label_col: str = "label",
    group_col: str = "scaffold_group",
    n_splits: int = 5,
    seed: int = RANDOM_STATE,
    target_test_size: float = TEST_SIZE,
) -> pd.Series:
    y = df[label_col].astype(int).values
    groups = df[group_col].astype(str).values
    overall_pos = y.mean()
    target_test_n = len(df) * target_test_size
    target_test_pos = y.sum() * target_test_size
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    best_mask = None
    best_score = None
    for _, test_idx in sgkf.split(df, y, groups=groups):
        mask = np.zeros(len(df), dtype=bool)
        mask[test_idx] = True
        test_n = int(mask.sum())
        test_pos = int(y[mask].sum())
        test_pos_rate = test_pos / test_n if test_n else 0.0
        size_penalty = abs(test_n - target_test_n) / max(len(df), 1)
        rate_penalty = abs(test_pos_rate - overall_pos)
        pos_count_penalty = abs(test_pos - target_test_pos) / max(y.sum(), 1)
        score = size_penalty + 2.0 * rate_penalty + 1.0 * pos_count_penalty
        if best_score is None or score < best_score:
            best_score = score
            best_mask = mask

    return pd.Series(np.where(best_mask, "test", "train"), index=df.index, name="scaffold_split_80_20")


def load_raw_endpoint(endpoint_key: str) -> pd.DataFrame:
    raw_name = ENDPOINTS[endpoint_key]["raw_name"]
    with zipfile.ZipFile(RAW_ZIP) as zz:
        path = f"fg_descriptor_analysis/{raw_name}/{raw_name}_canonical_analysis.csv"
        df = pd.read_csv(io.BytesIO(zz.read(path)), low_memory=False)
    df = df.loc[df["valid"].eq(1)].copy()
    df["SMILES"] = df["analysis_smiles"].astype(str)
    groups = [make_scaffold_group(s, m) for s, m in zip(df["SMILES"], df["murcko_scaffold"])]
    df["scaffold_group"] = [g for g, _ in groups]
    df["scaffold_group_type"] = [t for _, t in groups]
    return df


def merge_qm_if_needed(df: pd.DataFrame) -> pd.DataFrame:
    if not USE_QM or not QM_FILE:
        return df
    qm = pd.read_csv(QM_FILE)
    if MERGE_KEY not in df.columns or MERGE_KEY not in qm.columns:
        raise KeyError(f"MERGE_KEY '{MERGE_KEY}' must exist in both tables")
    if QM_COLS is None:
        exclude = {MERGE_KEY, "label", "No", "SMILES", "analysis_smiles"}
        qm_cols = [c for c in qm.columns if c not in exclude]
    else:
        qm_cols = list(QM_COLS)
    return df.merge(qm[[MERGE_KEY] + qm_cols], on=MERGE_KEY, how="left")


def build_endpoint_table(endpoint_key: str, rule_table: pd.DataFrame):
    df = load_raw_endpoint(endpoint_key)
    raw_name = ENDPOINTS[endpoint_key]["raw_name"]
    rules = rule_table.loc[rule_table["endpoint"].eq(raw_name)].copy()
    rule_cols = []
    for _, row in rules.iterrows():
        out_col = rule_col_name(row)
        df[out_col] = apply_rule(df[row["feature"]], row["mode"], float(row["threshold"]))
        rule_cols.append(out_col)

    df = merge_qm_if_needed(df)
    df["scaffold_split_80_20"] = choose_best_holdout(df)

    compact = list(ENDPOINTS[endpoint_key]["compact"])
    extended = dedupe_keep_order(compact + list(ENDPOINTS[endpoint_key]["extended_extra"]))
    union_features = list(extended)

    if USE_QM and QM_FILE:
        qm = pd.read_csv(QM_FILE)
        if QM_COLS is None:
            exclude = {MERGE_KEY, "label", "No", "SMILES", "analysis_smiles"}
            qm_use_cols = [c for c in qm.columns if c not in exclude]
        else:
            qm_use_cols = list(QM_COLS)
        union_features = dedupe_keep_order(union_features + qm_use_cols)

    keep_cols = dedupe_keep_order(ID_COLS + META_COLS + union_features)
    result = df[keep_cols].copy()
    feature_manifest = {
        "compact": compact,
        "extended": extended,
        "union": union_features,
        "all_rule_cols": rule_cols,
    }
    return result, feature_manifest


def endpoint_summary(df: pd.DataFrame, endpoint_key: str, feature_manifest: Dict[str, List[str]]):
    train_mask = df["scaffold_split_80_20"].eq("train")
    test_mask = df["scaffold_split_80_20"].eq("test")
    shared_groups = set(df.loc[train_mask, "scaffold_group"]) & set(df.loc[test_mask, "scaffold_group"])
    return {
        "endpoint": endpoint_key,
        "n_total": int(len(df)),
        "n_train": int(train_mask.sum()),
        "n_test": int(test_mask.sum()),
        "pos_rate_total": float(df["label"].mean()),
        "pos_rate_train": float(df.loc[train_mask, "label"].mean()),
        "pos_rate_test": float(df.loc[test_mask, "label"].mean()),
        "n_features_compact": len(feature_manifest["compact"]),
        "n_features_extended": len(feature_manifest["extended"]),
        "n_features_union": len(feature_manifest["union"]),
        "shared_scaffold_groups": len(shared_groups),
    }



## 파이프라인 실행

아래 셀 하나로 엔드포인트별 train/test 파일과 manifest를 모두 저장합니다.


In [5]:

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
rule_table = pd.read_csv(RULE_HIGHLIGHTS)

summaries = []
leakage_rows = []
manifest = {
    "settings": {
        "raw_zip": RAW_ZIP,
        "rule_highlights": RULE_HIGHLIGHTS,
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "use_qm": USE_QM,
        "qm_file": QM_FILE,
        "merge_key": MERGE_KEY,
        "qm_cols": QM_COLS,
    },
    "endpoints": {},
}

for endpoint_key in ENDPOINTS:
    df, feature_manifest = build_endpoint_table(endpoint_key, rule_table)

    combined_path = Path(OUTDIR) / f"{endpoint_key}_recommended_union_with_split.csv"
    train_path = Path(OUTDIR) / f"{endpoint_key}_recommended_union_train.csv"
    test_path = Path(OUTDIR) / f"{endpoint_key}_recommended_union_test.csv"
    assign_path = Path(OUTDIR) / f"{endpoint_key}_split_assignments.csv"

    df.to_csv(combined_path, index=False)
    df.loc[df["scaffold_split_80_20"].eq("train")].to_csv(train_path, index=False)
    df.loc[df["scaffold_split_80_20"].eq("test")].to_csv(test_path, index=False)
    df[ID_COLS + META_COLS].to_csv(assign_path, index=False)

    train_groups = set(df.loc[df["scaffold_split_80_20"].eq("train"), "scaffold_group"])
    test_groups = set(df.loc[df["scaffold_split_80_20"].eq("test"), "scaffold_group"])
    shared = train_groups & test_groups

    summaries.append(endpoint_summary(df, endpoint_key, feature_manifest))
    leakage_rows.append({
        "endpoint": endpoint_key,
        "shared_scaffold_groups": len(shared),
        "shared_example": next(iter(shared)) if shared else "",
    })

    manifest["endpoints"][endpoint_key] = {
        **feature_manifest,
        "combined": str(combined_path),
        "train": str(train_path),
        "test": str(test_path),
        "assignments": str(assign_path),
    }

summary_df = pd.DataFrame(summaries)
leakage_df = pd.DataFrame(leakage_rows)
summary_df.to_csv(Path(OUTDIR) / "split_summary.csv", index=False)
leakage_df.to_csv(Path(OUTDIR) / "leakage_check.csv", index=False)

with open(Path(OUTDIR) / "feature_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

(Path(OUTDIR) / "README.txt").write_text(
    "Unified scaffold-aware train/test pipeline outputs\n\n"
    "Use only these files for each endpoint:\n"
    "- *_recommended_union_train.csv\n"
    "- *_recommended_union_test.csv\n"
    "- feature_manifest.json\n",
    encoding="utf-8"
)

summary_df


[07:10:24] WARNING: not removing hydrogen atom without neighbors


[07:10:25] WARNING: not removing hydrogen atom without neighbors


,endpoint,n_total,n_train,n_test,pos_rate_total,pos_rate_train,pos_rate_test,n_features_compact,n_features_extended,n_features_union,shared_scaffold_groups
0,ames,13705,10964,2741,0.305290,0.305272,0.305363,13,34,34,0
1,invitro,1465,1170,295,0.111263,0.111111,0.111864,11,23,23,0
2,invivo,1932,1546,386,0.036749,0.036869,0.036269,10,19,19,0


## 누수 점검

In [6]:
leakage_df

,endpoint,shared_scaffold_groups,shared_example
0,ames,0,
1,invitro,0,
2,invivo,0,


## 출력 파일 확인

In [7]:

from pathlib import Path
sorted(str(p) for p in Path(OUTDIR).glob("*"))


['/mnt/data/unified_train_test_pipeline_outputs/README.txt',
 '/mnt/data/unified_train_test_pipeline_outputs/ames_recommended_union_test.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/ames_recommended_union_train.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/ames_recommended_union_with_split.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/ames_split_assignments.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/feature_manifest.json',
 '/mnt/data/unified_train_test_pipeline_outputs/invitro_recommended_union_test.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/invitro_recommended_union_train.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/invitro_recommended_union_with_split.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/invitro_split_assignments.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/invivo_recommended_union_test.csv',
 '/mnt/data/unified_train_test_pipeline_outputs/invivo_recommended_union_train.csv',
 '/mnt/data/unified_train_